In [ ]:
#train_model.py
"""
Comprehensive Model Training Pipeline for Multi-Disease Prediction Project
CS-245 Course Project: Health & Public Safety Intelligence Theme
Implements baseline, classical, and ensemble models with advanced evaluation
"""
import pandas as pd
import numpy as np
import os
import logging
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

# Machine Learning imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                            accuracy_score, precision_score, recall_score, f1_score,
                            roc_auc_score, confusion_matrix, classification_report)

# Baseline models
from sklearn.dummy import DummyRegressor, DummyClassifier

# Classical models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                             GradientBoostingRegressor, GradientBoostingClassifier)

# Advanced models
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.ensemble import (VotingRegressor, VotingClassifier,
                             StackingRegressor, StackingClassifier,
                             AdaBoostRegressor, AdaBoostClassifier)

# Feature selection
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


class MultiDiseaseModelTrainer:
    """
    Comprehensive model training pipeline for multi-disease prediction.
    Supports both regression (prevalence rates) and classification (high/low risk) tasks.
    """
    
    def __init__(self, data_path: str, task_type: str = 'regression', random_state: int = 42):
        """
        Initialize the model trainer.
        
        Args:
            data_path: Path to engineered features CSV
            task_type: 'regression' for prevalence rates, 'classification' for binary risk
            random_state: Random seed for reproducibility
        """
        self.data_path = data_path
        self.task_type = task_type
        self.random_state = random_state
        self.df = None
        self.features = None
        self.targets = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.models = {}
        self.results = {}
        self.best_models = {}
        self.scaler = StandardScaler()
        
        # Set task-specific configurations
        if task_type == 'regression':
            self.target_columns = ['Pct_Diabetes', 'Pct_HeartDisease', 'Pct_Stroke', 'Pct_Asthma']
            self.evaluation_metrics = ['mae', 'rmse', 'r2', 'mse']
        else:  # classification
            self.target_columns = ['HighRisk_Diabetes', 'HighRisk_HeartDisease', 
                                  'HighRisk_Stroke', 'HighRisk_Asthma']
            self.evaluation_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    
    def load_and_prepare_data(self, test_size: float = 0.2):
        """
        Load data and prepare for modeling.
        
        Args:
            test_size: Proportion of data to use for testing
        """
        logger.info(f"Loading data from {self.data_path}")
        
        if not os.path.exists(self.data_path):
            raise FileNotFoundError(f"Data file not found: {self.data_path}")
        
        # Load engineered features
        self.df = pd.read_csv(self.data_path)
        logger.info(f"Loaded dataset: {self.df.shape}")
        
        # Prepare target columns based on task type
        if self.task_type == 'classification':
            # Create binary classification targets from prevalence rates
            for disease in ['Diabetes', 'HeartDisease', 'Stroke', 'Asthma']:
                target_col = f'Pct_{disease}'
                if target_col in self.df.columns:
                    # High risk if prevalence > 75th percentile
                    threshold = self.df[target_col].quantile(0.75)
                    self.df[f'HighRisk_{disease}'] = (self.df[target_col] > threshold).astype(int)
        
        # Identify feature columns (exclude identifiers and targets)
        exclude_cols = ['FIPS', 'State', 'Sample_Size', 'County_Cluster']
        
        # Add all target columns to exclude
        all_possible_targets = (self.target_columns + 
                               [col for col in self.df.columns if col.startswith('Pct_')])
        exclude_cols.extend(all_possible_targets)
        
        # Get feature columns
        self.features = [col for col in self.df.columns if col not in exclude_cols]
        
        # Filter to only existing targets
        self.targets = [col for col in self.target_columns if col in self.df.columns]
        
        if not self.targets:
            raise ValueError(f"No target columns found: {self.target_columns}")
        
        logger.info(f"Using {len(self.features)} features")
        logger.info(f"Predicting {len(self.targets)} targets: {', '.join(self.targets)}")
        
        # Prepare X and y
        X = self.df[self.features].copy()
        y = self.df[self.targets].copy()
        
        # Handle missing values
        X = X.fillna(X.median())
        y = y.fillna(y.median())
        
        # Train-test split
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state
        )
        
        # Scale features
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)
        
        logger.info(f"Training set: {self.X_train.shape}")
        logger.info(f"Testing set: {self.X_test.shape}")
        
        # Save feature names for interpretation
        self.feature_names = self.features
        self.target_names = self.targets
    
    def create_baseline_models(self):
        """Create baseline models for comparison."""
        logger.info("Creating baseline models...")
        
        baseline_models = {}
        
        if self.task_type == 'regression':
            # Mean predictor
            baseline_models['Mean_Baseline'] = DummyRegressor(strategy='mean')
            
            # Median predictor
            baseline_models['Median_Baseline'] = DummyRegressor(strategy='median')
            
            # Constant predictor (predict 0)
            baseline_models['Constant_Baseline'] = DummyRegressor(strategy='constant', constant=0)
            
        else:  # classification
            # Most frequent class
            baseline_models['Most_Frequent'] = DummyClassifier(strategy='most_frequent')
            
            # Stratified random
            baseline_models['Stratified'] = DummyClassifier(strategy='stratified')
            
            # Uniform random
            baseline_models['Uniform'] = DummyClassifier(strategy='uniform')
        
        self.models['baseline'] = baseline_models
        return baseline_models
    
    def create_classical_models(self):
        """Create classical machine learning models."""
        logger.info("Creating classical ML models...")
        
        classical_models = {}
        
        if self.task_type == 'regression':
            # Linear models
            classical_models['Linear_Regression'] = LinearRegression()
            classical_models['Ridge_Regression'] = Ridge(alpha=1.0, random_state=self.random_state)
            classical_models['Lasso_Regression'] = Lasso(alpha=1.0, random_state=self.random_state)
            
            # Tree-based
            classical_models['Decision_Tree'] = DecisionTreeRegressor(
                max_depth=5, random_state=self.random_state
            )
            
            # SVM
            classical_models['SVM_RBF'] = SVR(kernel='rbf', C=1.0, epsilon=0.1)
            
            # KNN
            classical_models['KNN'] = KNeighborsRegressor(n_neighbors=5)
            
        else:  # classification
            # Linear models
            classical_models['Logistic_Regression'] = LogisticRegression(
                max_iter=1000, random_state=self.random_state
            )
            
            # Tree-based
            classical_models['Decision_Tree'] = DecisionTreeClassifier(
                max_depth=5, random_state=self.random_state
            )
            
            # SVM
            classical_models['SVM_RBF'] = SVC(kernel='rbf', C=1.0, probability=True, 
                                             random_state=self.random_state)
            
            # KNN
            classical_models['KNN'] = KNeighborsClassifier(n_neighbors=5)
            
            # Naive Bayes
            classical_models['Naive_Bayes'] = GaussianNB()
        
        self.models['classical'] = classical_models
        return classical_models
    
    def create_ensemble_models(self):
        """Create ensemble and advanced models."""
        logger.info("Creating ensemble models...")
        
        ensemble_models = {}
        
        if self.task_type == 'regression':
            # Random Forest
            ensemble_models['Random_Forest'] = RandomForestRegressor(
                n_estimators=100, max_depth=10, random_state=self.random_state, n_jobs=-1
            )
            
            # Gradient Boosting
            ensemble_models['Gradient_Boosting'] = GradientBoostingRegressor(
                n_estimators=100, learning_rate=0.1, max_depth=3,
                random_state=self.random_state
            )
            
            # XGBoost
            ensemble_models['XGBoost'] = XGBRegressor(
                n_estimators=100, learning_rate=0.1, max_depth=3,
                random_state=self.random_state, n_jobs=-1, verbosity=0
            )
            
            # LightGBM
            ensemble_models['LightGBM'] = LGBMRegressor(
                n_estimators=100, learning_rate=0.1, max_depth=5,
                random_state=self.random_state, n_jobs=-1, verbose=-1
            )
            
            # AdaBoost
            ensemble_models['AdaBoost'] = AdaBoostRegressor(
                n_estimators=50, learning_rate=1.0, random_state=self.random_state
            )
            
        else:  # classification
            # Random Forest
            ensemble_models['Random_Forest'] = RandomForestClassifier(
                n_estimators=100, max_depth=10, random_state=self.random_state, n_jobs=-1
            )
            
            # Gradient Boosting
            ensemble_models['Gradient_Boosting'] = GradientBoostingClassifier(
                n_estimators=100, learning_rate=0.1, max_depth=3,
                random_state=self.random_state
            )
            
            # XGBoost
            ensemble_models['XGBoost'] = XGBClassifier(
                n_estimators=100, learning_rate=0.1, max_depth=3,
                random_state=self.random_state, n_jobs=-1, verbosity=0
            )
            
            # LightGBM
            ensemble_models['LightGBM'] = LGBMClassifier(
                n_estimators=100, learning_rate=0.1, max_depth=5,
                random_state=self.random_state, n_jobs=-1, verbose=-1
            )
            
            # AdaBoost
            ensemble_models['AdaBoost'] = AdaBoostClassifier(
                n_estimators=50, learning_rate=1.0, random_state=self.random_state
            )
        
        self.models['ensemble'] = ensemble_models
        return ensemble_models
    
    def create_stacking_ensemble(self, base_estimators=None):
        """
        Create a stacking ensemble model.
        
        Args:
            base_estimators: List of (name, estimator) tuples for base learners
        """
        logger.info("Creating stacking ensemble...")
        
        if base_estimators is None:
            if self.task_type == 'regression':
                base_estimators = [
                    ('rf', RandomForestRegressor(n_estimators=50, random_state=self.random_state)),
                    ('gb', GradientBoostingRegressor(n_estimators=50, random_state=self.random_state)),
                    ('xgb', XGBRegressor(n_estimators=50, random_state=self.random_state, verbosity=0))
                ]
                final_estimator = LinearRegression()
            else:
                base_estimators = [
                    ('rf', RandomForestClassifier(n_estimators=50, random_state=self.random_state)),
                    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=self.random_state)),
                    ('xgb', XGBClassifier(n_estimators=50, random_state=self.random_state, verbosity=0))
                ]
                final_estimator = LogisticRegression(random_state=self.random_state)
        
        if self.task_type == 'regression':
            stacking_model = StackingRegressor(
                estimators=base_estimators,
                final_estimator=final_estimator,
                cv=5,
                n_jobs=-1
            )
        else:
            stacking_model = StackingClassifier(
                estimators=base_estimators,
                final_estimator=final_estimator,
                cv=5,
                n_jobs=-1
            )
        
        self.models['stacking'] = {'Stacking_Ensemble': stacking_model}
        return stacking_model
    
    def train_single_target_model(self, model, target_idx, model_name):
        """
        Train a model for a single target.
        
        Args:
            model: Model instance
            target_idx: Index of target to train for
            model_name: Name of the model
            
        Returns:
            Trained model and predictions
        """
        # Get target data
        y_train_target = self.y_train.iloc[:, target_idx]
        y_test_target = self.y_test.iloc[:, target_idx]
        
        # Train model
        model.fit(self.X_train_scaled, y_train_target)
        
        # Make predictions
        y_pred_train = model.predict(self.X_train_scaled)
        y_pred_test = model.predict(self.X_test_scaled)
        
        # Calculate metrics
        metrics = self._calculate_metrics(y_test_target, y_pred_test, y_train_target, y_pred_train)
        
        return model, y_pred_test, metrics
    
    def train_multi_target_model(self, model, model_name):
        """
        Train a model for all targets simultaneously.
        
        Args:
            model: Model instance
            model_name: Name of the model
            
        Returns:
            Trained model and predictions
        """
        # Train model
        model.fit(self.X_train_scaled, self.y_train)
        
        # Make predictions
        y_pred_train = model.predict(self.X_train_scaled)
        y_pred_test = model.predict(self.X_test_scaled)
        
        # Calculate metrics for each target
        all_metrics = {}
        for i, target_name in enumerate(self.target_names):
            if self.task_type == 'regression':
                y_test_target = self.y_test.iloc[:, i]
                y_pred_test_target = y_pred_test[:, i] if y_pred_test.ndim > 1 else y_pred_test
                y_train_target = self.y_train.iloc[:, i]
                y_pred_train_target = y_pred_train[:, i] if y_pred_train.ndim > 1 else y_pred_train
            else:
                y_test_target = self.y_test.iloc[:, i]
                y_pred_test_target = y_pred_test[:, i] if y_pred_test.ndim > 1 else y_pred_test
                y_train_target = self.y_train.iloc[:, i]
                y_pred_train_target = y_pred_train[:, i] if y_pred_train.ndim > 1 else y_pred_train
            
            metrics = self._calculate_metrics(y_test_target, y_pred_test_target,
                                             y_train_target, y_pred_train_target)
            all_metrics[target_name] = metrics
        
        # Calculate average metrics
        avg_metrics = self._calculate_average_metrics(all_metrics)
        
        return model, y_pred_test, all_metrics, avg_metrics
    
    def _calculate_metrics(self, y_true, y_pred, y_true_train=None, y_pred_train=None):
        """Calculate evaluation metrics."""
        metrics = {}
        
        if self.task_type == 'regression':
            # Regression metrics
            metrics['mae'] = mean_absolute_error(y_true, y_pred)
            metrics['rmse'] = np.sqrt(mean_squared_error(y_true, y_pred))
            metrics['mse'] = mean_squared_error(y_true, y_pred)
            metrics['r2'] = r2_score(y_true, y_pred)
            
            if y_true_train is not None and y_pred_train is not None:
                metrics['r2_train'] = r2_score(y_true_train, y_pred_train)
                metrics['train_test_r2_diff'] = metrics['r2_train'] - metrics['r2']
        
        else:  # classification
            # Convert probabilities to predictions if needed
            if y_pred.ndim > 1 and y_pred.shape[1] > 1:
                y_pred_binary = np.argmax(y_pred, axis=1)
                y_pred_proba = y_pred[:, 1] if y_pred.shape[1] == 2 else y_pred
            else:
                y_pred_binary = (y_pred > 0.5).astype(int)
                y_pred_proba = y_pred
            
            # Classification metrics
            metrics['accuracy'] = accuracy_score(y_true, y_pred_binary)
            metrics['precision'] = precision_score(y_true, y_pred_binary, average='weighted', zero_division=0)
            metrics['recall'] = recall_score(y_true, y_pred_binary, average='weighted', zero_division=0)
            metrics['f1'] = f1_score(y_true, y_pred_binary, average='weighted', zero_division=0)
            
            # ROC-AUC (handle binary and multiclass)
            try:
                if len(np.unique(y_true)) > 2:
                    metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='weighted')
                else:
                    metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba)
            except:
                metrics['roc_auc'] = 0.5  # Default for problematic cases
        
        return metrics
    
    def _calculate_average_metrics(self, all_metrics):
        """Calculate average metrics across all targets."""
        avg_metrics = {}
        
        # Get all metric names from first target
        first_target = list(all_metrics.keys())[0]
        metric_names = all_metrics[first_target].keys()
        
        for metric in metric_names:
            values = [all_metrics[target][metric] for target in all_metrics.keys()]
            avg_metrics[f'avg_{metric}'] = np.mean(values)
            avg_metrics[f'std_{metric}'] = np.std(values)
        
        return avg_metrics
    
    def perform_feature_selection(self, k: int = 20):
        """
        Perform feature selection using mutual information.
        
        Args:
            k: Number of top features to select
            
        Returns:
            Selected feature indices and names
        """
        logger.info(f"Selecting top {k} features using mutual information...")
        
        if self.task_type == 'regression':
            selector = SelectKBest(score_func=mutual_info_regression, k=min(k, self.X_train.shape[1]))
        else:
            selector = SelectKBest(score_func=mutual_info_classification, k=min(k, self.X_train.shape[1]))
        
        selector.fit(self.X_train_scaled, self.y_train.iloc[:, 0])  # Use first target
        
        # Get selected feature indices
        selected_indices = selector.get_support(indices=True)
        selected_features = [self.feature_names[i] for i in selected_indices]
        
        logger.info(f"Selected {len(selected_features)} features")
        logger.info(f"Top 10 features: {selected_features[:10]}")
        
        return selected_indices, selected_features
    
    def perform_hyperparameter_tuning(self, model, param_grid, cv=5):
        """
        Perform hyperparameter tuning using GridSearchCV.
        
        Args:
            model: Model instance
            param_grid: Parameter grid for tuning
            cv: Number of cross-validation folds
            
        Returns:
            Best model and tuning results
        """
        logger.info("Performing hyperparameter tuning...")
        
        grid_search = GridSearchCV(
            model,
            param_grid,
            cv=cv,
            scoring='r2' if self.task_type == 'regression' else 'f1_weighted',
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(self.X_train_scaled, self.y_train.iloc[:, 0])
        
        logger.info(f"Best parameters: {grid_search.best_params_}")
        logger.info(f"Best score: {grid_search.best_score_:.4f}")
        
        return grid_search.best_estimator_, grid_search.best_params_
    
    def train_all_models(self, use_cv: bool = False, cv_folds: int = 5):
        """
        Train all models and store results.
        
        Args:
            use_cv: Whether to use cross-validation
            cv_folds: Number of cross-validation folds
        """
        logger.info("=" * 70)
        logger.info("STARTING MODEL TRAINING")
        logger.info("=" * 70)
        
        # Initialize models if not already done
        if not self.models:
            self.create_baseline_models()
            self.create_classical_models()
            self.create_ensemble_models()
            self.create_stacking_ensemble()
        
        # Dictionary to store all results
        self.results = {
            'model_comparison': {},
            'target_results': {target: {} for target in self.target_names},
            'training_info': {
                'task_type': self.task_type,
                'n_features': len(self.features),
                'n_train_samples': len(self.X_train),
                'n_test_samples': len(self.X_test),
                'target_names': self.target_names
            }
        }
        
        # Train models for each category
        for category in ['baseline', 'classical', 'ensemble', 'stacking']:
            if category in self.models:
                logger.info(f"\nTraining {category} models...")
                
                for model_name, model in self.models[category].items():
                    logger.info(f"  Training {model_name}...")
                    
                    try:
                        if self.task_type == 'regression':
                            # For regression, train multi-target model
                            trained_model, y_pred, target_metrics, avg_metrics = self.train_multi_target_model(
                                model, model_name
                            )
                            
                            # Store results
                            self.results['model_comparison'][model_name] = avg_metrics
                            
                            # Store model
                            self.models[category][model_name] = trained_model
                            
                            # Store predictions for later analysis
                            if 'predictions' not in self.results:
                                self.results['predictions'] = {}
                            self.results['predictions'][model_name] = y_pred
                            
                            # Store target-specific results
                            for target_name, metrics in target_metrics.items():
                                self.results['target_results'][target_name][model_name] = metrics
                            
                        else:
                            # For classification, we need to handle each target separately
                            all_target_metrics = {}
                            trained_models = {}
                            
                            for i, target_name in enumerate(self.target_names):
                                # Clone model for each target
                                model_clone = model.__class__(**model.get_params())
                                trained_model, y_pred, metrics = self.train_single_target_model(
                                    model_clone, i, model_name
                                )
                                
                                trained_models[target_name] = trained_model
                                all_target_metrics[target_name] = metrics
                            
                            # Calculate average metrics
                            avg_metrics = self._calculate_average_metrics(all_target_metrics)
                            
                            # Store results
                            self.results['model_comparison'][model_name] = avg_metrics
                            self.models[category][model_name] = trained_models
                            
                            # Store target-specific results
                            for target_name, metrics in all_target_metrics.items():
                                self.results['target_results'][target_name][model_name] = metrics
                        
                        logger.info(f"    Completed {model_name}")
                        
                    except Exception as e:
                        logger.error(f"    Failed to train {model_name}: {str(e)}")
        
        logger.info("\n" + "=" * 70)
        logger.info("MODEL TRAINING COMPLETE")
        logger.info("=" * 70)
        
        # Identify best model for each target
        self._identify_best_models()
        
        return self.results
    
    def _identify_best_models(self):
        """Identify the best model for each target and overall."""
        self.best_models = {'overall': {}, 'by_target': {}}
        
        # Determine primary metric for comparison
        if self.task_type == 'regression':
            primary_metric = 'avg_r2'  # Higher is better
            metric_direction = 'maximize'
        else:
            primary_metric = 'avg_f1'  # Higher is better
            metric_direction = 'maximize'
        
        # Find best overall model
        best_overall_score = -float('inf') if metric_direction == 'maximize' else float('inf')
        best_overall_model = None
        
        for model_name, metrics in self.results['model_comparison'].items():
            if primary_metric in metrics:
                score = metrics[primary_metric]
                
                if metric_direction == 'maximize':
                    if score > best_overall_score:
                        best_overall_score = score
                        best_overall_model = model_name
                else:
                    if score < best_overall_score:
                        best_overall_score = score
                        best_overall_model = model_name
        
        if best_overall_model:
            self.best_models['overall'] = {
                'model': best_overall_model,
                'score': best_overall_score,
                'metric': primary_metric
            }
            logger.info(f"Best overall model: {best_overall_model} ({primary_metric}: {best_overall_score:.4f})")
        
        # Find best model for each target
        for target_name in self.target_names:
            target_results = self.results['target_results'][target_name]
            
            if not target_results:
                continue
            
            best_score = -float('inf') if metric_direction == 'maximize' else float('inf')
            best_model = None
            
            for model_name, metrics in target_results.items():
                if primary_metric.replace('avg_', '') in metrics:
                    score = metrics[primary_metric.replace('avg_', '')]
                    
                    if metric_direction == 'maximize':
                        if score > best_score:
                            best_score = score
                            best_model = model_name
                    else:
                        if score < best_score:
                            best_score = score
                            best_model = model_name
            
            if best_model:
                self.best_models['by_target'][target_name] = {
                    'model': best_model,
                    'score': best_score,
                    'metric': primary_metric.replace('avg_', '')
                }
                logger.info(f"Best model for {target_name}: {best_model} (score: {best_score:.4f})")
    
    def evaluate_model_performance(self):
        """Generate comprehensive performance evaluation report."""
        logger.info("\n" + "=" * 70)
        logger.info("MODEL PERFORMANCE EVALUATION")
        logger.info("=" * 70)
        
        # Create comparison DataFrame
        comparison_data = []
        
        for model_name, metrics in self.results['model_comparison'].items():
            row = {'Model': model_name}
            row.update(metrics)
            comparison_data.append(row)
        
        comparison_df = pd.DataFrame(comparison_data)
        
        # Sort by primary metric
        if self.task_type == 'regression':
            primary_metric = 'avg_r2'
        else:
            primary_metric = 'avg_f1'
        
        if primary_metric in comparison_df.columns:
            comparison_df = comparison_df.sort_values(primary_metric, ascending=False)
        
        logger.info("\nModel Comparison (sorted by performance):")
        print(comparison_df.to_string(index=False))
        
        # Save comparison results
        os.makedirs('reports/model_evaluation', exist_ok=True)
        comparison_df.to_csv('reports/model_evaluation/model_comparison.csv', index=False)
        
        # Generate detailed report for each target
        for target_name in self.target_names:
            target_comparison = []
            target_results = self.results['target_results'][target_name]
            
            for model_name, metrics in target_results.items():
                row = {'Model': model_name}
                row.update(metrics)
                target_comparison.append(row)
            
            if target_comparison:
                target_df = pd.DataFrame(target_comparison)
                
                # Sort by primary metric
                primary_target_metric = 'r2' if self.task_type == 'regression' else 'f1'
                if primary_target_metric in target_df.columns:
                    target_df = target_df.sort_values(primary_target_metric, ascending=False)
                
                # Save target-specific results
                target_df.to_csv(f'reports/model_evaluation/{target_name}_performance.csv', index=False)
        
        return comparison_df
    
    def analyze_feature_importance(self, top_n: int = 20):
        """
        Analyze feature importance from tree-based models.
        
        Args:
            top_n: Number of top features to analyze
        """
        logger.info(f"\nAnalyzing feature importance (top {top_n} features)...")
        
        os.makedirs('reports/feature_importance', exist_ok=True)
        
        # Focus on ensemble models (they have feature_importances_ attribute)
        ensemble_models = ['Random_Forest', 'Gradient_Boosting', 'XGBoost', 'LightGBM']
        
        for model_name in ensemble_models:
            if model_name in self.models.get('ensemble', {}):
                try:
                    # Get the trained model
                    if self.task_type == 'classification':
                        # For classification, we have separate models for each target
                        # Use first target's model for analysis
                        first_target = list(self.models['ensemble'][model_name].keys())[0]
                        model = self.models['ensemble'][model_name][first_target]
                    else:
                        model = self.models['ensemble'][model_name]
                    
                    # Get feature importances
                    if hasattr(model, 'feature_importances_'):
                        importances = model.feature_importances_
                        
                        # Create DataFrame
                        importance_df = pd.DataFrame({
                            'Feature': self.feature_names,
                            'Importance': importances
                        }).sort_values('Importance', ascending=False).head(top_n)
                        
                        # Save to CSV
                        importance_df.to_csv(f'reports/feature_importance/{model_name}_importance.csv', index=False)
                        
                        # Create visualization
                        plt.figure(figsize=(10, 6))
                        plt.barh(range(len(importance_df)), importance_df['Importance'])
                        plt.yticks(range(len(importance_df)), importance_df['Feature'])
                        plt.xlabel('Feature Importance')
                        plt.title(f'Top {top_n} Features - {model_name}')
                        plt.gca().invert_yaxis()
                        plt.tight_layout()
                        plt.savefig(f'reports/feature_importance/{model_name}_importance.png', dpi=150, bbox_inches='tight')
                        plt.close()
                        
                        logger.info(f"  {model_name}: Top 3 features - {', '.join(importance_df['Feature'].head(3).tolist())}")
                    
                except Exception as e:
                    logger.warning(f"  Could not analyze feature importance for {model_name}: {e}")
    
    def create_model_interpretation_report(self):
        """Create comprehensive model interpretation report."""
        logger.info("\nCreating model interpretation report...")
        
        report_dir = 'reports/model_interpretation'
        os.makedirs(report_dir, exist_ok=True)
        
        report_content = """
        ============================================
        MODEL INTERPRETATION REPORT
        ============================================
        
        Project: Multi-Disease Prediction using BRFSS and EPA AQI Data
        Task Type: {task_type}
        Number of Features: {n_features}
        Number of Targets: {n_targets}
        
        """.format(
            task_type=self.task_type.upper(),
            n_features=len(self.feature_names),
            n_targets=len(self.target_names)
        )
        
        # Add best models information
        report_content += "\nBEST MODELS IDENTIFIED:\n"
        report_content += "-" * 50 + "\n"
        
        if self.best_models.get('overall'):
            best_overall = self.best_models['overall']
            report_content += f"\nOverall Best Model: {best_overall['model']}\n"
            report_content += f"Performance ({best_overall['metric']}): {best_overall['score']:.4f}\n"
        
        if self.best_models.get('by_target'):
            report_content += "\nBest Models by Target:\n"
            for target, info in self.best_models['by_target'].items():
                report_content += f"  {target}: {info['model']} ({info['metric']}: {info['score']:.4f})\n"
        
        # Add key insights
        report_content += "\n\nKEY INSIGHTS:\n"
        report_content += "-" * 50 + "\n"
        
        # Analyze model performance patterns
        if 'model_comparison' in self.results:
            comparison_df = pd.DataFrame(self.results['model_comparison']).T
            
            if self.task_type == 'regression':
                # Find models with best R2
                if 'avg_r2' in comparison_df.columns:
                    top_3_r2 = comparison_df.nlargest(3, 'avg_r2')
                    report_content += "\nTop 3 Models by R² Score:\n"
                    for idx, row in top_3_r2.iterrows():
                        report_content += f"  {idx}: R² = {row['avg_r2']:.4f}, MAE = {row['avg_mae']:.4f}\n"
                
                # Analyze overfitting
                if 'avg_train_test_r2_diff' in comparison_df.columns:
                    overfit_models = comparison_df[comparison_df['avg_train_test_r2_diff'] > 0.2]
                    if not overfit_models.empty:
                        report_content += "\nModels with Potential Overfitting (train-test R² diff > 0.2):\n"
                        for idx, row in overfit_models.iterrows():
                            report_content += f"  {idx}: Diff = {row['avg_train_test_r2_diff']:.4f}\n"
            
            else:  # classification
                if 'avg_f1' in comparison_df.columns:
                    top_3_f1 = comparison_df.nlargest(3, 'avg_f1')
                    report_content += "\nTop 3 Models by F1 Score:\n"
                    for idx, row in top_3_f1.iterrows():
                        report_content += f"  {idx}: F1 = {row['avg_f1']:.4f}, Accuracy = {row['avg_accuracy']:.4f}\n"
        
        # Add recommendations
        report_content += "\n\nRECOMMENDATIONS:\n"
        report_content += "-" * 50 + "\n"
        
        if self.best_models.get('overall'):
            best_model = self.best_models['overall']['model']
            
            report_content += f"\n1. Primary Recommendation: Use {best_model} as the production model.\n"
            
            if 'Stacking' in best_model:
                report_content += "   - Stacking ensembles combine multiple models for robust predictions\n"
                report_content += "   - Consider deployment complexity and inference time\n"
            elif 'XGBoost' in best_model or 'LightGBM' in best_model:
                report_content += "   - Gradient boosting models often provide excellent performance\n"
                report_content += "   - Good balance between accuracy and interpretability\n"
            elif 'Random_Forest' in best_model:
                report_content += "   - Random Forest provides good performance with built-in feature importance\n"
                report_content += "   - Less prone to overfitting than individual trees\n"
        
        report_content += "\n2. Model Deployment Considerations:\n"
        report_content += "   - For web applications: Consider model size and inference speed\n"
        report_content += "   - For batch predictions: Can use more complex ensemble models\n"
        report_content += "   - Ensure proper monitoring of model performance over time\n"
        
        report_content += "\n3. Future Improvements:\n"
        report_content += "   - Incorporate temporal data for trend analysis\n"
        report_content += "   - Add more granular geographic data (census tract level)\n"
        report_content += "   - Include additional environmental factors (water quality, noise pollution)\n"
        report_content += "   - Implement model retraining pipeline with new data\n"
        
        # Save report
        report_path = os.path.join(report_dir, 'model_interpretation_report.txt')
        with open(report_path, 'w') as f:
            f.write(report_content)
        
        logger.info(f"Model interpretation report saved to {report_path}")
        
        return report_content
    
    def save_models(self, output_dir: str = 'models'):
        """
        Save trained models to disk.
        
        Args:
            output_dir: Directory to save models
        """
        logger.info(f"\nSaving models to {output_dir}...")
        
        os.makedirs(output_dir, exist_ok=True)
        
        # Save all models
        for category, models in self.models.items():
            category_dir = os.path.join(output_dir, category)
            os.makedirs(category_dir, exist_ok=True)
            
            for model_name, model in models.items():
                if model is not None:
                    model_path = os.path.join(category_dir, f'{model_name}.pkl')
                    
                    try:
                        # Handle different model structures
                        if isinstance(model, dict):
                            # Multiple models (one per target)
                            joblib.dump(model, model_path)
                        else:
                            # Single model
                            joblib.dump(model, model_path)
                        
                        logger.info(f"  Saved {category}/{model_name}")
                    except Exception as e:
                        logger.error(f"  Failed to save {model_name}: {e}")
        
        # Save scaler
        scaler_path = os.path.join(output_dir, 'scaler.pkl')
        joblib.dump(self.scaler, scaler_path)
        
        # Save feature names
        features_info = {
            'feature_names': self.feature_names,
            'target_names': self.target_names,
            'task_type': self.task_type
        }
        
        features_path = os.path.join(output_dir, 'features_info.json')
        with open(features_path, 'w') as f:
            json.dump(features_info, f, indent=2)
        
        # Save results
        results_path = os.path.join(output_dir, 'training_results.json')
        
        # Convert numpy arrays to lists for JSON serialization
        results_serializable = {}
        for key, value in self.results.items():
            if isinstance(value, dict):
                results_serializable[key] = {}
                for subkey, subvalue in value.items():
                    if isinstance(subvalue, np.ndarray):
                        results_serializable[key][subkey] = subvalue.tolist()
                    elif isinstance(subvalue, dict):
                        # Handle nested dicts
                        results_serializable[key][subkey] = {}
                        for k, v in subvalue.items():
                            if isinstance(v, np.ndarray):
                                results_serializable[key][subkey][k] = v.tolist()
                            else:
                                results_serializable[key][subkey][k] = v
                    else:
                        results_serializable[key][subkey] = subvalue
            else:
                results_serializable[key] = value
        
        with open(results_path, 'w') as f:
            json.dump(results_serializable, f, indent=2)
        
        # Save best models info
        best_models_path = os.path.join(output_dir, 'best_models.json')
        with open(best_models_path, 'w') as f:
            json.dump(self.best_models, f, indent=2)
        
        logger.info(f"All models and artifacts saved to {output_dir}/")
    
    def generate_training_summary(self):
        """Generate comprehensive training summary report."""
        logger.info("\n" + "=" * 70)
        logger.info("TRAINING SUMMARY")
        logger.info("=" * 70)
        
        summary = f"""
        MULTI-DISEASE PREDICTION MODEL TRAINING SUMMARY
        =================================================
        
        Dataset Information:
        - Total features: {len(self.feature_names)}
        - Training samples: {self.X_train.shape[0]}
        - Testing samples: {self.X_test.shape[0]}
        - Task type: {self.task_type.upper()}
        
        Target Variables:
        {', '.join(self.target_names)}
        
        Models Trained:
        - Baseline models: {len(self.models.get('baseline', {}))}
        - Classical models: {len(self.models.get('classical', {}))}
        - Ensemble models: {len(self.models.get('ensemble', {}))}
        - Stacking ensemble: {len(self.models.get('stacking', {}))}
        
        Best Overall Model:
        """
        
        if self.best_models.get('overall'):
            best = self.best_models['overall']
            summary += f"- Model: {best['model']}\n"
            summary += f"- Performance ({best['metric']}): {best['score']:.4f}\n"
        
        summary += "\nOutput Files Generated:\n"
        summary += "- models/: Saved model files (.pkl)\n"
        summary += "- reports/model_evaluation/: Performance metrics\n"
        summary += "- reports/feature_importance/: Feature importance analysis\n"
        summary += "- reports/model_interpretation/: Model interpretation reports\n"
        
        # Print summary
        print(summary)
        
        # Save summary
        summary_path = 'reports/training_summary.txt'
        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
        
        with open(summary_path, 'w') as f:
            f.write(summary)
        
        logger.info(f"Training summary saved to {summary_path}")
        
        return summary


def mutual_info_classification(X, y):
    """Wrapper for mutual_info_classif that handles the function signature."""
    from sklearn.feature_selection import mutual_info_classif
    return mutual_info_classif(X, y, random_state=42)


def main():
    """Main function to execute the complete model training pipeline."""
    logger.info("=" * 70)
    logger.info("MULTI-DISEASE PREDICTION: MODEL TRAINING PIPELINE")
    logger.info("=" * 70)
    
    # Configuration
    FEATURES_DATA = 'data/processed/engineered_features.csv'
    TASK_TYPE = 'regression'  # 'regression' or 'classification'
    TEST_SIZE = 0.2
    RANDOM_STATE = 42
    
    # Create output directories
    os.makedirs('models', exist_ok=True)
    os.makedirs('reports/model_evaluation', exist_ok=True)
    os.makedirs('reports/feature_importance', exist_ok=True)
    os.makedirs('reports/model_interpretation', exist_ok=True)
    
    try:
        # Initialize model trainer
        trainer = MultiDiseaseModelTrainer(
            data_path=FEATURES_DATA,
            task_type=TASK_TYPE,
            random_state=RANDOM_STATE
        )
        
        # Step 1: Load and prepare data
        logger.info("\n[STEP 1] Loading and preparing data...")
        trainer.load_and_prepare_data(test_size=TEST_SIZE)
        
        # Step 2: Train all models
        logger.info("\n[STEP 2] Training models...")
        results = trainer.train_all_models(use_cv=False, cv_folds=5)
        
        # Step 3: Evaluate performance
        logger.info("\n[STEP 3] Evaluating model performance...")
        comparison_df = trainer.evaluate_model_performance()
        
        # Step 4: Analyze feature importance
        logger.info("\n[STEP 4] Analyzing feature importance...")
        trainer.analyze_feature_importance(top_n=20)
        
        # Step 5: Create interpretation report
        logger.info("\n[STEP 5] Creating model interpretation report...")
        trainer.create_model_interpretation_report()
        
        # Step 6: Save models
        logger.info("\n[STEP 6] Saving models...")
        trainer.save_models()
        
        # Step 7: Generate summary
        logger.info("\n[STEP 7] Generating training summary...")
        trainer.generate_training_summary()
        
        # Print key results
        logger.info("\n" + "=" * 70)
        logger.info("TRAINING PIPELINE COMPLETE - KEY RESULTS")
        logger.info("=" * 70)
        
        if trainer.best_models.get('overall'):
            best = trainer.best_models['overall']
            logger.info(f"Best Overall Model: {best['model']}")
            logger.info(f"Performance Score: {best['score']:.4f} ({best['metric']})")
        
        logger.info(f"\nDetailed results saved in 'reports/' directory")
        logger.info(f"Models saved in 'models/' directory")
        
        return trainer
        
    except Exception as e:
        logger.error(f"Model training pipeline failed: {e}")
        import traceback
        traceback.print_exc()
        return None


if __name__ == "__main__":
    # Run the training pipeline
    trainer = main()

2025-12-14 15:08:12,174 - INFO - ======================================================================
2025-12-14 15:08:12,174 - INFO - MULTI-DISEASE PREDICTION: MODEL TRAINING PIPELINE
2025-12-14 15:08:12,175 - INFO - ======================================================================
2025-12-14 15:08:12,176 - INFO - 
[STEP 1] Loading and preparing data...
2025-12-14 15:08:12,176 - INFO - Loading data from data/processed/engineered_features.csv
2025-12-14 15:08:12,184 - INFO - Loaded dataset: (52, 91)
2025-12-14 15:08:12,185 - INFO - Columns: ['HighBP_Risk', 'HighChol_Risk', 'Smoking_Risk', 'Diabetes_Risk', 'Cardiovascular_Risk_Index', 'Obesity_Indicator', 'Hypertension_Indicator', 'HighChol_Indicator', 'Metabolic_Syndrome_Score', 'Smoking_Resp_Risk', 'AQI_Resp_Risk', 'Respiratory_Risk_Score', 'Diabetes_Hypertension_Interaction', 'CVD_Diabetes_Interaction', 'Physical_Activity_Score', 'Diet_Quality_Score', 'Non_Smoking_Score', 'Moderate_Drinking_Score', 'Healthy_Lifestyle_Index', '

            Model  avg_mae  std_mae  avg_rmse  std_rmse  avg_mse  std_mse    avg_r2   std_r2  avg_r2_train  std_r2_train  avg_train_test_r2_diff  std_train_test_r2_diff
 Ridge_Regression 0.299799 0.108251  0.429413  0.131145 0.201594 0.107942  0.902803 0.066014      0.992543      0.007459                0.089740                0.059768
          XGBoost 0.539261 0.265400  0.827065  0.459650 0.895316 0.935031  0.657323 0.256437      0.999954      0.000026                0.342631                0.256426
Linear_Regression 0.396520 0.336116  0.565419  0.513956 0.583850 0.802040  0.555717 0.714383      1.000000      0.000000                0.444283                0.714383
    Decision_Tree 0.871084 0.269120  1.149118  0.299845 1.410379 0.736772  0.461897 0.331638      0.937605      0.030632                0.475708                0.321614
    Random_Forest 0.720949 0.313144  1.201206  0.749821 2.005127 2.445761  0.370631 0.552969      0.941323      0.038421                0.570692           

In [ ]:
# Optional: Create visualizations
if trainer:
    try:
        # Create performance comparison visualization
        import matplotlib.pyplot as plt
        
        # Get comparison DataFrame
        comparison_data = []
        for model_name, metrics in trainer.results['model_comparison'].items():
            row = {'Model': model_name}
            row.update(metrics)
            comparison_data.append(row)
        
        comparison_df = pd.DataFrame(comparison_data)
        
        # Create bar chart of top models
        if TASK_TYPE == 'regression':
            metric_col = 'avg_r2'
            metric_name = 'R² Score'
        else:
            metric_col = 'avg_f1'
            metric_name = 'F1 Score'
        
        if metric_col in comparison_df.columns:
            # Get top 10 models
            top_models = comparison_df.nlargest(10, metric_col)
            
            plt.figure(figsize=(12, 6))
            bars = plt.barh(range(len(top_models)), top_models[metric_col])
            plt.yticks(range(len(top_models)), top_models['Model'])
            plt.xlabel(metric_name)
            plt.title(f'Top 10 Models by {metric_name}')
            plt.gca().invert_yaxis()
            
            # Add value labels
            for i, (bar, val) in enumerate(zip(bars, top_models[metric_col])):
                plt.text(val + 0.01, i, f'{val:.3f}', va='center')
            
            plt.tight_layout()
            plt.savefig('reports/model_evaluation/top_models_comparison.png', dpi=150, bbox_inches='tight')
            plt.close()
            
            logger.info("Created model comparison visualization")
    
    except Exception as e:
        logger.warning(f"Could not create visualizations: {e}")